# Train one YOLO model on Colab

Trains a single model variant end-to-end and persists the run folder to Google Drive.
Run one model per Colab session (free tier ~12h is not enough for all variants at once).

**Disconnect protection:** during training, `weights/last.pt`, `results.csv`, and
`args.yaml` are synced to Drive every `CHECKPOINT_EVERY` epochs (default 10). If Colab
disconnects mid-training, at most that many epochs of work are lost. Reopen the
notebook, set `RESUME = True`, and the trainer picks up from the last synced epoch.

Before running: upload `dataset.zip` + `dataset.meta.json` to
`MyDrive/yolo-pipeline/datasets/v1/`.

In [ ]:
# === EDIT THESE PER SESSION ===
REPO_URL    = "https://github.com/tahmid013/yolo.git"
REPO_BRANCH = "main"
DATASET_VERSION = "v1"

# v11 already trained by the reference team (imported via analysis/import_reference.py).
# This pipeline is now focused on training the v12 family. Cycle through these 4:
#   yolo12n (start here) -> yolo12s -> yolo12m -> yolo12l
MODEL       = "yolo12n"
CONFIG      = "configs/yolo12n.yaml"
RESUME      = False                      # True to continue the latest Drive run for MODEL
CHECKPOINT_EVERY = 10                    # sync weights/results to Drive every N epochs (0 to disable)
# ==============================

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi

In [ ]:
import shutil, os
if os.path.isdir('/content/code'):
    shutil.rmtree('/content/code')
!git clone --branch {REPO_BRANCH} {REPO_URL} /content/code
%cd /content/code
!git rev-parse HEAD

In [ ]:
!pip install -q -r requirements.txt
import ultralytics, torch
print('ultralytics', ultralytics.__version__, '| torch', torch.__version__, '| cuda', torch.version.cuda)

In [ ]:
from pipeline.dataset import ensure_dataset
from pipeline import paths

data_yaml = ensure_dataset(
    zip_path=paths.dataset_zip(DATASET_VERSION),
    meta_path=paths.dataset_meta(DATASET_VERSION),
    target=paths.LOCAL_DATASET,
)
print('data.yaml:', data_yaml)

In [ ]:
from pipeline.persist import cleanup_tmp
cleanup_tmp(paths.RUNS_DIR)

In [ ]:
from pipeline.train import run as train_run
run_dir = train_run(
    model=MODEL,
    config=CONFIG,
    drive_runs_dir=paths.RUNS_DIR,
    local_runs_dir=paths.LOCAL_RUNS,
    data_yaml=data_yaml,
    dataset_meta_path=paths.dataset_meta(DATASET_VERSION),
    base_config='configs/base.yaml',
    resume=RESUME,
    checkpoint_every=CHECKPOINT_EVERY,
)
print('Drive run dir:', run_dir)

In [ ]:
from pipeline.evaluate import run as eval_run
eval_payload = eval_run(
    run_dir=run_dir,
    data_yaml=data_yaml,
    drive_runs_dir=paths.RUNS_DIR,
)
print('test mAP50:', eval_payload['overall']['mAP50'])
print('test mAP50-95:', eval_payload['overall']['mAP50_95'])